**MGMT298D: Science and Strategy of AI**

# Week 6B: Transformers (Custom + Pre-trained)

# 1 Setup

#### Import libraries for transformers, datasets, and model evaluation.

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling1D

from datasets import load_dataset, Dataset
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import torch
import matplotlib.pyplot as plt

---
# 2 Custom Transformer on AG News

# 2.1 Load & Prepare Data

#### Load the AG News dataset from huggingface/datasets with 120k articles in four categories. Tokenize using Keras' TextVectorization layer.

In [ ]:
# Load AG News dataset
dataset = load_dataset('ag_news')
CLASS_NAMES = ['World', 'Sports', 'Business', 'Sci/Tech']

print(f"Training samples: {len(dataset['train']):,}")
print(f"Test samples:     {len(dataset['test']):,}")
print(f"Classes:          {CLASS_NAMES}")
print(f"\nSample articles:")
for i in range(3):
    label = CLASS_NAMES[dataset['train'][i]['label']]
    text = dataset['train'][i]['text'][:100]
    print(f"  [{label}] {text}...")

In [ ]:
# Use a subset for faster training
TRAIN_SIZE = 20000
TEST_SIZE = 4000
VOCAB_SIZE = 15000
MAX_LEN = 200

# Extract text and labels
train_texts = dataset['train']['text'][:TRAIN_SIZE]
train_labels = np.array(dataset['train']['label'][:TRAIN_SIZE])
test_texts = dataset['test']['text'][:TEST_SIZE]
test_labels = np.array(dataset['test']['label'][:TEST_SIZE])

# Create Keras TextVectorization layer for tokenization
vectorizer = layers.TextVectorization(max_tokens=VOCAB_SIZE, output_sequence_length=MAX_LEN)
vectorizer.adapt(train_texts)

# Vectorize text
x_train = vectorizer(np.array(train_texts)).numpy()
x_test = vectorizer(np.array(test_texts)).numpy()
y_train = train_labels
y_test = test_labels

print(f"Training set: {x_train.shape}")
print(f"Test set:     {x_test.shape}")
print(f"Vocabulary:   {VOCAB_SIZE:,} tokens")
print(f"Sequence len: {MAX_LEN}")

# 2.2 Transformer Block

#### Build a TransformerBlock with multi-head attention, feed-forward layers, residual connections, and layer normalization. This is the core component of models like BERT.

In [ ]:
# Custom Transformer Block
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads)
        self.ffn = keras.Sequential([
            Dense(ff_dim, activation='relu'),
            Dense(embed_dim),
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = Dropout(rate)
        self.drop2 = Dropout(rate)

    def call(self, inputs, training):
        attn = self.att(inputs, inputs)
        attn = self.drop1(attn, training=training)
        out1 = self.norm1(inputs + attn)       # Residual + LayerNorm
        ffn = self.ffn(out1)
        ffn = self.drop2(ffn, training=training)
        return self.norm2(out1 + ffn)          # Residual + LayerNorm


# Token + Positional Embedding
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        return self.token_emb(x) + self.pos_emb(positions)


print("Defined: TransformerBlock (MultiHeadAttention + FFN + Residuals)")
print("Defined: TokenAndPositionEmbedding (Token + Positional)")

# 2.3 Train & Evaluate

#### Build and train transformer models with one and three blocks. Compare performance to show how depth affects accuracy and parameter count.

In [ ]:
# Build transformer classifier with configurable depth
EMBED_DIM = 64
NUM_HEADS = 2
FF_DIM = 64

def build_transformer_classifier(num_blocks=1):
    inputs = layers.Input(shape=(MAX_LEN,))
    x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBED_DIM)(inputs)
    for _ in range(num_blocks):
        x = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM, rate=0.1)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    x = Dense(32, activation='relu')(x)
    outputs = Dense(4, activation='softmax')(x)
    return keras.Model(inputs=inputs, outputs=outputs)


# --- Train 1-block model ---
model_1block = build_transformer_classifier(num_blocks=1)
model_1block.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Training 1-block Transformer...")
history_1b = model_1block.fit(x_train, y_train, batch_size=64, epochs=10,
                              validation_split=0.1, verbose=0)
acc_1b = model_1block.evaluate(x_test, y_test, verbose=0)[1]
print(f"1-Block Test Accuracy: {acc_1b:.4f}  |  Parameters: {model_1block.count_params():,}")


# --- Train 3-block model ---
model_3block = build_transformer_classifier(num_blocks=3)
model_3block.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("\nTraining 3-block Transformer...")
history_3b = model_3block.fit(x_train, y_train, batch_size=64, epochs=10,
                              validation_split=0.1, verbose=0)
acc_3b = model_3block.evaluate(x_test, y_test, verbose=0)[1]
print(f"3-Block Test Accuracy: {acc_3b:.4f}  |  Parameters: {model_3block.count_params():,}")

In [ ]:
# Print training summary
print("\n=== Custom Transformer Training Summary ===")
print(f"1-Block final train loss: {history_1b.history['loss'][-1]:.4f}")
print(f"1-Block final val loss:   {history_1b.history['val_loss'][-1]:.4f}")
print(f"\n3-Block final train loss: {history_3b.history['loss'][-1]:.4f}")
print(f"3-Block final val loss:   {history_3b.history['val_loss'][-1]:.4f}")

In [ ]:
# Evaluate best custom model
best_custom = model_3block if acc_3b >= acc_1b else model_1block
best_custom_name = '3-Block' if acc_3b >= acc_1b else '1-Block'
best_custom_acc = max(acc_1b, acc_3b)

y_pred_custom = np.argmax(best_custom.predict(x_test, verbose=0), axis=1)
cm = confusion_matrix(y_test, y_pred_custom)

print(f"\n=== Confusion Matrix — {best_custom_name} Custom Transformer ===")
print(f"\n{'':12s} {CLASS_NAMES[0]:10s} {CLASS_NAMES[1]:10s} {CLASS_NAMES[2]:10s} {CLASS_NAMES[3]:10s}")
for i, class_name in enumerate(CLASS_NAMES):
    row_str = f"{class_name:12s}"
    for j in range(4):
        row_str += f" {cm[i, j]:10d}"
    print(row_str)

print("\n" + classification_report(y_test, y_pred_custom, target_names=CLASS_NAMES))

---
# 3 Pre-trained Transformer — Transfer Learning

# 3.1 Zero-Shot Classification

#### Use a pre-trained BART model for zero-shot classification without fine-tuning. Describe categories in natural language and match articles to the best label.

In [ ]:
# Load zero-shot classification pipeline
print("Loading zero-shot classifier (facebook/bart-large-mnli)...")
zs_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli",
                          device=0 if tf.config.list_physical_devices('GPU') else -1)

# Classify a sample of test articles
ZS_SAMPLE = 200
sample_texts = dataset['test']['text'][:ZS_SAMPLE]
sample_labels = dataset['test']['label'][:ZS_SAMPLE]
candidate_labels = ['world news', 'sports', 'business', 'science and technology']

print(f"Running zero-shot on {ZS_SAMPLE} test samples...\n")
zs_preds = []
for text in sample_texts:
    result = zs_classifier(text, candidate_labels)
    pred_idx = candidate_labels.index(result['labels'][0])
    zs_preds.append(pred_idx)

acc_zs = accuracy_score(sample_labels, zs_preds)
print(f"Zero-Shot Accuracy ({ZS_SAMPLE} samples): {acc_zs:.4f}")
print("\nReminder: this model has NEVER been trained on AG News!")

In [ ]:
# Show a few zero-shot predictions
print("\nSample Zero-Shot Predictions:\n")
for i in range(5):
    true = CLASS_NAMES[sample_labels[i]]
    pred = CLASS_NAMES[zs_preds[i]]
    match = '✓' if sample_labels[i] == zs_preds[i] else '✗'
    print(f"  {match} True: {true:10s} | Pred: {pred:10s} | {sample_texts[i][:80]}...")

# 3.2 Fine-Tune DistilBERT

#### Fine-tune a pre-trained DistilBERT model on AG News. Combine language knowledge from pre-training with task-specific supervision using the same training data size as the custom transformer.

In [ ]:
# Prepare HuggingFace datasets for fine-tuning
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

# Use same subset sizes as custom transformer for fair comparison
train_ds = Dataset.from_dict({
    'text': dataset['train']['text'][:TRAIN_SIZE],
    'label': dataset['train']['label'][:TRAIN_SIZE]
})

test_ds = Dataset.from_dict({
    'text': dataset['test']['text'][:TEST_SIZE],
    'label': dataset['test']['label'][:TEST_SIZE]
})

def tokenize_fn(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

print(f"Tokenized train: {len(train_ds):,} samples")
print(f"Tokenized test:  {len(test_ds):,} samples")

In [ ]:
# Load pre-trained DistilBERT and add classification head
model_ft = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=4)

# Training configuration
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy='epoch',
    logging_strategy='epoch',
    save_strategy='no',
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to='none',
)

trainer = Trainer(
    model=model_ft,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
)

print("Fine-tuning DistilBERT (3 epochs)...")
trainer.train()

In [ ]:
# Generate predictions on test set
ft_predictions = trainer.predict(test_ds)
y_pred_ft = np.argmax(ft_predictions.predictions, axis=1)
acc_ft = accuracy_score(test_ds['label'], y_pred_ft)

print(f"Fine-Tuned DistilBERT Test Accuracy: {acc_ft:.4f}")
print(f"Parameters: {model_ft.num_parameters():,}")

In [ ]:
# Confusion matrix for fine-tuned model
cm_ft = confusion_matrix(test_ds['label'], y_pred_ft)

print("\n=== Confusion Matrix — Fine-Tuned DistilBERT ===")
print(f"\n{'':12s} {CLASS_NAMES[0]:10s} {CLASS_NAMES[1]:10s} {CLASS_NAMES[2]:10s} {CLASS_NAMES[3]:10s}")
for i, class_name in enumerate(CLASS_NAMES):
    row_str = f"{class_name:12s}"
    for j in range(4):
        row_str += f" {cm_ft[i, j]:10d}"
    print(row_str)

print("\n" + classification_report(test_ds['label'], y_pred_ft, target_names=CLASS_NAMES))

# 3.3 Final Model Comparison

#### Compare all four models: custom transformers, zero-shot classification, and fine-tuned DistilBERT. Show the dramatic advantage of pre-training on massive text corpora.

In [ ]:
# Compare all models
print("\n" + "="*70)
print("AG NEWS CLASSIFICATION — FINAL MODEL COMPARISON")
print("="*70)

models_info = [
    ('1-Block Transformer', acc_1b, model_1block.count_params()),
    ('3-Block Transformer', acc_3b, model_3block.count_params()),
    (f'Zero-Shot BART (n={ZS_SAMPLE})', acc_zs, None),
    ('Fine-Tuned DistilBERT', acc_ft, model_ft.num_parameters()),
]

print(f"\n{'Model':<30s} {'Test Accuracy':>15s} {'Parameters':>20s}")
print("-" * 70)

for name, acc, params in models_info:
    if params is not None:
        params_str = f"{params:,}"
    else:
        params_str = "(pre-trained)"
    print(f"{name:<30s} {acc:>15.4f} {params_str:>20s}")

print("-" * 70)
from_scratch_best = max(acc_1b, acc_3b)
improvement = (acc_ft - from_scratch_best) * 100

print(f"\nKey Insight:")
print(f"  From-scratch best:     {from_scratch_best:.4f}")
print(f"  Fine-tuned BERT:       {acc_ft:.4f}")
print(f"  Improvement:           {improvement:+.1f} percentage points")
print(f"\nPre-training on massive text corpora gives DistilBERT a huge advantage.")
print(f"It learns transferable language knowledge that fine-tuning leverages quickly.")

In [ ]:
# Plot all model accuracies
model_names = ['1-Block', '3-Block', f'Zero-Shot\nBART', 'Fine-Tuned\nDistilBERT']
accuracies = [acc_1b, acc_3b, acc_zs, acc_ft]
colors = ['lightcoral', 'coral', 'lightskyblue', 'lightgreen']

plt.figure(figsize=(10, 6))
bars = plt.bar(model_names, accuracies, color=colors, edgecolor='black', linewidth=1.5)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('AG News Classification: All Models Compared', fontsize=14, fontweight='bold')
plt.ylim([0, 1])
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{acc:.4f}', ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()